# CALL CENTER AGENT PERFORMANCE ANALYSIS

Analyze 16 public simulated banking calls using CPU transcription, structured extraction and evidence checks. Run the sections in order. Store `GEMINI_API_KEY` in Colab Secrets and enable notebook access.

Research and selection details are in `research/Research_Survey.xlsx` and `README.md`.

## 1. Project setup

Extract the project files. Existing results are preserved. Changed source files are backed up before replacement.

In [ ]:
# Extract source files and verify the embedded package.
from pathlib import Path
import base64, hashlib, io, zipfile, json, subprocess, sys, os
PROJECT = Path('/content/contact_center_clean_start') if Path('/content').exists() else Path.cwd() / 'contact_center_clean_start'
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '26b93538c677becb8d4200f045ac24d6aee90524c43d9866e374ee4465d6de16'
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        target = (PROJECT / name).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe archive path')
        data = archive.read(name)
        if target.exists() and target.read_bytes() != data:
            if name.startswith('results/'):
                continue
            backup = PROJECT / '.previous_source' / name
            backup.parent.mkdir(parents=True, exist_ok=True)
            backup.write_bytes(target.read_bytes())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print('Project:', PROJECT)
print('Research:', PROJECT / 'research/Research_Survey.xlsx')

## 2. Dependencies and tests

Use a separate environment and run the regression tests.

In [ ]:
# Install dependencies outside the notebook kernel.
VENV = PROJECT / '.venv'
if not (VENV / 'bin/python').exists():
    created = subprocess.run([sys.executable, '-m', 'venv', str(VENV)])
    if created.returncode != 0:
        # Some Colab images omit ensurepip; bootstrap virtualenv outside the kernel.
        tool_dir = PROJECT / '.venv_bootstrap'
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--target', str(tool_dir), 'virtualenv'], check=True)
        tool_env = os.environ.copy()
        tool_env['PYTHONPATH'] = str(tool_dir)
        subprocess.run([sys.executable, '-m', 'virtualenv', str(VENV)], env=tool_env, check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements.txt')], check=True)

def run_stage(*args, env=None):
    subprocess.run([PYTHON, *args], cwd=PROJECT, env=env, check=True)

run_stage('-m', 'unittest', 'discover', '-s', 'tests', '-v')

## 3. Data selection and audit

Download a pinned sample of two calls per scenario. Verify source hashes, channel format and reference completeness.

In [ ]:
# Download and audit the selected calls.
run_stage('run.py', 'prepare')
manifest = json.loads((PROJECT/'data_source/selection.json').read_text())
audit = json.loads((PROJECT/'results/audit.json').read_text())
print('Population:', manifest['population_calls'])
print('Selected:', len(manifest['selection']), '| Audio channels:', len(audit))
from IPython.display import display
import pandas as pd
display(pd.DataFrame(manifest['selection']))
display(pd.DataFrame(audit).drop(columns=['sha256']).head())

## 4. Transcription

Decode full speaker channels with Whisper base.en on CPU. Reuse matching transcripts from this project cache.

In [ ]:
# Transcribe each complete speaker channel.
run_stage('run.py', 'asr')

## 5. ASR evaluation

Calculate WER against complete human references. Compare the supplied machine transcripts on the same channels.

In [ ]:
# Compare ASR outputs on the same reference subset.
report = json.loads((PROJECT/'results/wer_summary.json').read_text())
print(json.dumps(report, indent=2))
wer_rows = json.loads((PROJECT/'results/wer_channels.json').read_text())
display(pd.DataFrame(wer_rows).sort_values('wer', ascending=False).head(6))

## 6. API pilot

Analyze one call and inspect its citations. The API key is passed through memory, not written to project files.

In [ ]:
# Load the key without printing it.
MODEL = 'gemini-3.5-flash-lite'
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except ImportError:
    from getpass import getpass
    api_key = os.environ.get('GEMINI_API_KEY') or getpass('Gemini API key: ')
if not api_key:
    raise RuntimeError('Add GEMINI_API_KEY to Colab Secrets and enable notebook access.')
api_env = os.environ.copy()
api_env['GEMINI_API_KEY'] = api_key
run_stage('run.py', 'analyze', '--limit', '1', '--model', MODEL, env=api_env)
pilot = json.loads((PROJECT/'results/analysis.json').read_text())
print(pilot[0]['summary'])
print('Review:', pilot[0]['review_reasons'])
display(pilot[0]['tasks'])

## 7. Call analysis

Process all 16 calls. Successful requests are cached. Failed requests stop the batch without automatic retries.

In [ ]:
# Resume cached calls and analyze the remaining sample.
run_stage('run.py', 'analyze', '--limit', '16', '--model', MODEL, env=api_env)
results = json.loads((PROJECT/'results/analysis.json').read_text())
display(pd.DataFrame([{'call_id':r['call_id'], 'sentiment':r['sentiment']['label'], 'needs_review':r['needs_review'], 'summary':r['summary']} for r in results]))
print(json.loads((PROJECT/'results/api_status.json').read_text()))

## 8. Results interface

Browse saved summaries, citations and review flags. Use section 8.1 if audio playback fails in the interface.

In [ ]:
# Start the read-only interface.
import time, urllib.request
if 'ui_process' not in globals() or ui_process.poll() is not None:
    ui_log = (PROJECT/'results/ui.log').open('w')
    ui_process = subprocess.Popen([PYTHON, '-u', 'app.py'], cwd=PROJECT, stdout=ui_log, stderr=subprocess.STDOUT)
ready = False
for _ in range(60):
    if ui_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:7860/', timeout=1) as r:
            ready = r.status == 200
        if ready: break
    except Exception:
        time.sleep(1)
if not ready:
    print((PROJECT/'results/ui.log').read_text())
    raise RuntimeError('UI did not start; see the log above.')
try:
    from google.colab import output
    url = output.eval_js('google.colab.kernel.proxyPort(7860)')
    from IPython.display import HTML
    display(HTML(f'<a href="{url}" target="_blank">Open call review interface</a>'))
except ImportError:
    print('Open http://127.0.0.1:7860')

## 8.1. Audio playback

Play the selected call directly in the notebook. This bypasses the interface audio URL without rerunning ASR or analysis.

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

CALL_ID = "ada8e8780baa4a5a"  # Change this ID to review another call.

for folder, label in [("agent", "Agent"), ("caller", "Customer")]:
    path = (
        Path(PROJECT) / "data_source" / "data" / "audio"
        / folder / f"{CALL_ID}.wav"
    )
    print(f"\n{label}: {path}")

    if not path.is_file():
        print("Audio file not found.")
        continue

    print(f"File size: {path.stat().st_size:,} bytes")
    try:
        # Embed WAV bytes directly, avoiding Gradio's audio URL.
        display(Audio(data=path.read_bytes(), embed=True))
    except Exception as error:
        print(f"{type(error).__name__}: {error}")


## 9. Manual review

Fill `results/manual_review.csv` after listening. Enter 0 or 1 and a reviewer name. Blank entries remain unscored.

In [ ]:
# Compute metrics from explicit human labels.
run_stage('evaluate_review.py')

## 10. Export

Download the project and current results without audio, credentials, caches or runtime dependencies. Include this notebook and the presentation recording when submitting.

In [ ]:
# Export source files and current results.
submission = PROJECT.parent / 'CALL_CENTER_AGENT_PERFORMANCE_ANALYSIS.zip'
with zipfile.ZipFile(submission, 'w', zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(PROJECT.rglob('*')):
        relative = p.relative_to(PROJECT)
        if not p.is_file() or any(part in {'.venv', '.venv_bootstrap', '.previous_source', 'data_source', 'cache', '__pycache__'} for part in relative.parts):
            continue
        if p.name.endswith('.log') or p.name == '.env':
            continue
        archive.write(p, Path('contact_center_clean')/relative)
print(submission)
try:
    from google.colab import files
    files.download(str(submission))
except ImportError:
    pass